# Entrenamiento del modelo de segmentación sobre FloorPlanCAD

In [ ]:
!nvidia-smi

In [ ]:
%pip install ultralytics==8.4.173 albumentations==2.0.8

Define rutas y parámetros del entrenamiento.

In [ ]:
from pathlib import Path

REPO = Path.cwd().parents[1]
DATA = REPO / "datasets/floor-plan-cad-yolo/floor-plan-cad-yolo.yaml"
RUNS = REPO / "runs"

MODEL = "yolo26m-seg.pt"
IMGSZ = 720
BATCH = 4
EPOCHS = 100
SEED = 42
NAME = Path(MODEL).stem

assert DATA.exists(), f"No existe el dataset: {DATA}"

Define las augmentations que simulan planos analógicos escaneados.

In [ ]:
import albumentations as A

# Degradaciones de escaneo; Ultralytics las aplica solo en entrenamiento
AUGMENTATIONS = [
    A.RandomBrightnessContrast(p=0.5),                         # tono del papel
    A.RandomGamma(p=0.3),
    A.Illumination(p=0.3),                                     # luz despareja
    A.Spatter(mode="mud", p=0.3),                              # manchas
    A.OneOf([A.Morphological(scale=2, operation="erosion"),    # trazo más grueso
             A.Morphological(scale=2, operation="dilation")],  # trazo más fino
            p=0.3),
    A.GaussNoise(std_range=(0.02, 0.1), p=0.3),                # ruido de escaneo
    A.SaltAndPepper(amount=(0.001, 0.01), p=0.2),              # motas
    A.Defocus(radius=(1, 3), p=0.2),                           # desenfoque
    A.ImageCompression(quality_range=(40, 90), p=0.3),         # artefactos JPG
    A.ToGray(p=1.0),                                           # BLD-AR está en escala de grises
]

Entrena el modelo con semilla fija y las augmentations definidas.

In [ ]:
from ultralytics import YOLO

model = YOLO(MODEL)
model.train(
    data=str(DATA), imgsz=IMGSZ, batch=BATCH, epochs=EPOCHS,
    seed=SEED, deterministic=True,
    project=str(RUNS), name=NAME,
    hsv_h=0.0, hsv_s=0.0, hsv_v=0.4,  # planos en gris: sin tinte, solo brillo
    degrees=3.0,                      # escaneo levemente torcido
    flipud=0.5, fliplr=0.5,           # el plano no tiene arriba ni abajo
    mosaic=1.0, scale=0.5,
    augmentations=AUGMENTATIONS,
)
RUN_DIR = Path(model.trainer.save_dir)
print(RUN_DIR)

Evalúa el mejor modelo sobre el split de test de FloorPlanCAD.

In [ ]:
best = YOLO(RUN_DIR / "weights" / "best.pt")
metrics = best.val(data=str(DATA), split="test", imgsz=IMGSZ, batch=BATCH,
                   project=str(RUN_DIR), name="test")

Muestra las métricas de test en una tabla

In [ ]:
from IPython.display import Markdown, display

rows = [("Box", metrics.box.map50, metrics.box.map),
        ("Seg", metrics.seg.map50, metrics.seg.map)]
display(Markdown(
    f"**Test · {MODEL} · imgsz={IMGSZ}**\n\n| | mAP50 | mAP50-95 |\n|---|---|---|\n"
    + "\n".join(f"| {name} | {m50:.3f} | {m:.3f} |" for name, m50, m in rows)))